# Baseline shared-UMAP 2×2 grid

Generates two PNGs for the defense deck (slide 14):

1. **Scatter grid** — 4 panels showing baseline embeddings on:
   - LC25000 (in-dist, 5 classes)
   - LC25000 + NCT-CRC (close-shift colon)
   - LC25000 + Chaoyang (further-shift colon)
   - LC25000 + LungHist700 (lung)

2. **Thumbnails version** — same 4 panels with tissue-image thumbnails overlaid on top of each cluster.

All 4 panels share **one UMAP fit** so cluster positions are directly comparable across datasets.

Run all cells top-to-bottom. Total time ~15-20 min in Colab (image encoding is the bottleneck).

## 0 — Bootstrap: clone repo, install deps

In [ ]:
# --- Cell 0: bootstrap ---
import os, sys, subprocess
os.environ["KERAS_BACKEND"] = "tensorflow"

REPO_URL = "https://github.com/lcandau/histopathology-clip-lab.git"
REPO_DIR = "/content/histopathology-clip-lab"
BRANCH   = "main"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
        print(f"Cloned {BRANCH}")
    else:
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=False)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=False)
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
    sys.path.insert(0, REPO_DIR)

    !pip install -q keras keras-hub tensorflow \
        transformers==4.46.0 umap-learn kagglehub h5py

    from google.colab import drive, files
    drive.mount('/content/drive')

print('Bootstrap done.')

## 1 — Imports

In [ ]:
# --- Cell 1: imports ---
import os, json, math, shutil, urllib.request, zipfile, re
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from matplotlib.patches import Patch
from PIL import Image

import tensorflow as tf
import keras
import keras_hub

from src.utils.paths import is_colab, repo_root, drive_root, results_dir
from src.utils.repro import set_global_seed, enable_op_determinism
from src.data.lc25000 import CLASS_INFO, ID_TO_INDEX, INDEX_TO_NAME

## 2 — Config

In [ ]:
# --- Cell 2: config (BASELINE only) ---
from src.utils.paths import run_dir

SEED       = 42
IMG_SIZE   = 224
MAX_LEN    = 24
EMBED_DIM  = 256
INIT_TEMP  = 0.07
BATCH_SIZE = 64

set_global_seed(SEED)
enable_op_determinism()

# Baseline checkpoint — same path used by every OOD eval notebook
CHECKPOINT_PATH = run_dir('exp_01_baseline', 'baseline') / 'weights.weights.h5'
print(f'Checkpoint: {CHECKPOINT_PATH}')
if not CHECKPOINT_PATH.is_file():
    # Fallback: search Drive for the baseline .weights.h5
    candidates = list((drive_root() / 'exp_01_baseline').rglob('*.weights.h5'))
    print(f'  fallback candidates under exp_01_baseline: {candidates}')
    if candidates:
        CHECKPOINT_PATH = candidates[0]
        print(f'  using {CHECKPOINT_PATH}')
assert CHECKPOINT_PATH.is_file(), (
    f'Baseline checkpoint not found. Please point CHECKPOINT_PATH to your baseline .weights.h5.'
)

## 3 — Build baseline model + load checkpoint

In [ ]:
# --- Cell 3: model construction (baseline) ---
image_backbone = keras_hub.models.Backbone.from_preset('resnet_50_imagenet')
image_backbone.trainable = False
text_backbone = keras_hub.models.Backbone.from_preset('bert_base_en_uncased')
text_backbone.trainable = False
text_pre = keras_hub.models.BertTextClassifierPreprocessor.from_preset(
    'bert_base_en_uncased', sequence_length=MAX_LEN,
)

def _l2(x, axis=-1):
    return tf.math.l2_normalize(x, axis=axis)

class CLIPModel_HF(keras.Model):
    def __init__(self, img_backbone, text_backbone, embed_dim=EMBED_DIM, init_temp=INIT_TEMP, **kwargs):
        super().__init__(**kwargs)
        self.img_backbone  = img_backbone
        self.text_backbone = text_backbone
        self.img_projection = keras.Sequential([
            keras.layers.Dense(embed_dim, use_bias=False, dtype='float32'),
            keras.layers.LayerNormalization(dtype='float32'),
        ], name='img_projection')
        self.text_projection = keras.Sequential([
            keras.layers.Dense(embed_dim, use_bias=False, dtype='float32'),
            keras.layers.LayerNormalization(dtype='float32'),
        ], name='text_projection')
        self.logit_scale = self.add_weight(
            name='logit_scale', shape=(),
            initializer=keras.initializers.Constant(math.log(1.0 / init_temp)),
            trainable=True, dtype='float32',
        )
    def call(self, inputs, training=False):
        """Full forward pass — required for `model.built = True` before load_weights."""
        image, text_tokens = inputs
        v = self.encode_image(image, training=training)
        t = self.encode_text(text_tokens, training=training)
        return v, t
    def encode_image(self, images, training=False):
        # keras_hub ResNet50 returns a 4D feature map (B, H, W, C) — GAP to 2D before projecting.
        # Same pattern as the exp_07 OOD eval notebooks; without this, the Dense/LN projection
        # keeps the spatial dims and downstream numpy.concatenate fails.
        feats = self.img_backbone(images, training=False)
        if isinstance(feats, dict):
            feats = feats.get('pooled_output', list(feats.values())[0])
        if feats.shape.rank == 4:
            feats = tf.reduce_mean(feats, axis=[1, 2])
        return _l2(self.img_projection(feats, training=training))
    def encode_text(self, token_dict, training=False):
        out = self.text_backbone(token_dict)
        pooled = out.get('pooled_output', out) if isinstance(out, dict) else out
        return _l2(self.text_projection(pooled, training=training))

model = CLIPModel_HF(image_backbone, text_backbone)
# Trigger outer __call__ so `model.built` becomes True
dummy_img = tf.zeros((1, IMG_SIZE, IMG_SIZE, 3))
dummy_txt = text_pre(['A histopathology image of colon adenocarcinoma.'])
v_dummy, t_dummy = model((dummy_img, dummy_txt))
print(f'model.built = {model.built}  ·  image feat shape = {v_dummy.shape}  ·  text feat shape = {t_dummy.shape}')
assert v_dummy.shape.rank == 2 and v_dummy.shape[-1] == EMBED_DIM, \
    f'Expected (B, {EMBED_DIM}) image features, got {v_dummy.shape}'
model.load_weights(str(CHECKPOINT_PATH))
print('Model built and checkpoint loaded.')

## 4 — Prepare LC25000 test paths

Uses the split JSON saved by the training notebook (seed 42).

In [ ]:
# --- Cell 4: LC25000 test split paths ---
import kagglehub
from src.data.lc25000 import discover_records, stratified_split, save_split, load_split

lc_root = Path(kagglehub.dataset_download('andrewmvd/lung-and-colon-cancer-histopathological-images'))
print(f'LC25000 root: {lc_root}')

LC_SPLIT_PATH = results_dir('splits') / f'lc25000_seed{SEED}.json'

if LC_SPLIT_PATH.exists():
    lc_split = load_split(LC_SPLIT_PATH)
    print(f'Loaded LC25000 split from {LC_SPLIT_PATH}')
else:
    print('Building LC25000 split from scratch...')
    all_paths, all_indices = discover_records(lc_root)
    lc_split = stratified_split(all_paths, all_indices,
                                val_fraction=0.10, test_fraction=0.10, seed=SEED)
    LC_SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)
    save_split(lc_split, LC_SPLIT_PATH)
    print(f'Saved split to {LC_SPLIT_PATH}')

lc_test_paths   = np.asarray(lc_split['test_paths'])
lc_test_indices = np.asarray(lc_split['test_indices'], dtype=np.int32)
print(f'LC25000 test set: {len(lc_test_paths)} images')
for idx in range(len(CLASS_INFO)):
    print(f'  [{idx}] {INDEX_TO_NAME[idx]}: {int((lc_test_indices == idx).sum())}')

## 5 — Prepare NCT-CRC-HE-7K paths

Downloads from Zenodo (cached to Drive on first run).

In [ ]:
# --- Cell 5: NCT paths ---
NCT_URL  = 'https://zenodo.org/records/1214456/files/CRC-VAL-HE-7K.zip'
NCT_NAME = 'CRC-VAL-HE-7K'
DRIVE_NCT_ZIP = drive_root() / 'ood_datasets' / f'{NCT_NAME}.zip'
LOCAL_NCT_DIR = Path('/content/ood_data') / NCT_NAME

if not LOCAL_NCT_DIR.exists():
    LOCAL_NCT_DIR.parent.mkdir(parents=True, exist_ok=True)
    if DRIVE_NCT_ZIP.exists():
        print(f'Copying {DRIVE_NCT_ZIP} → local')
        shutil.copy(DRIVE_NCT_ZIP, f'/content/{NCT_NAME}.zip')
    else:
        print(f'Downloading {NCT_URL}')
        DRIVE_NCT_ZIP.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(NCT_URL, DRIVE_NCT_ZIP)
        shutil.copy(DRIVE_NCT_ZIP, f'/content/{NCT_NAME}.zip')
    with zipfile.ZipFile(f'/content/{NCT_NAME}.zip') as z:
        z.extractall('/content/ood_data')

NCT_TO_LC25000 = {'NORM': 'benign colon tissue', 'TUM': 'colon adenocarcinoma'}
nct_paths, nct_lc_indices = [], []
for cls, lc_name in NCT_TO_LC25000.items():
    lc_id = next(c.id for c in CLASS_INFO if c.name == lc_name)
    for f in sorted((LOCAL_NCT_DIR / cls).iterdir()):
        if f.suffix.lower() in {'.tif', '.tiff', '.png', '.jpg', '.jpeg'}:
            nct_paths.append(str(f))
            nct_lc_indices.append(ID_TO_INDEX[lc_id])
print(f'NCT-CRC: {len(nct_paths)} patches')

## 6 — Prepare Chaoyang paths

Reads from `~/MyDrive/TFM/datasets/Chaoyang` (test split JSON).

In [ ]:
# --- Cell 6: Chaoyang paths ---
# Layout on Drive (from the working OOD eval notebook):
#   /content/drive/MyDrive/TFM/datasets/Chaoyang/chaoyang-data/{train,test}/*.JPG
#   /content/drive/MyDrive/TFM/datasets/Chaoyang/chaoyang-data/{train,test}.json
# Each JSON entry has {"label": int, "name": "test/<file>.JPG"} — the split folder
# is already part of the relative name.
CHAOYANG_ROOT      = Path('/content/drive/MyDrive/TFM/datasets/Chaoyang')
CHAOYANG_DATA_ROOT = CHAOYANG_ROOT / 'chaoyang-data'
assert CHAOYANG_DATA_ROOT.exists(), f'Chaoyang not found at {CHAOYANG_DATA_ROOT}'

CHAOYANG_TO_LC25000 = {0: 'benign colon tissue', 2: 'colon adenocarcinoma'}

records = []
for split in ('train', 'test'):
    jp = CHAOYANG_DATA_ROOT / f'{split}.json'
    if jp.exists():
        records.extend(json.loads(jp.read_text()))
print(f'Chaoyang JSON records (train+test): {len(records)}')

chao_paths, chao_lc_indices = [], []
for rec in records:
    lbl = int(rec.get('label', -1))
    if lbl not in CHAOYANG_TO_LC25000: continue
    rel = rec.get('name') or rec.get('filename')
    if not rel: continue
    p = CHAOYANG_DATA_ROOT / rel
    if p.is_file():
        lc_name = CHAOYANG_TO_LC25000[lbl]
        lc_id = next(c.id for c in CLASS_INFO if c.name == lc_name)
        chao_paths.append(str(p))
        chao_lc_indices.append(ID_TO_INDEX[lc_id])
print(f'Chaoyang: {len(chao_paths)} patches (normal + adenocarcinoma)')

## 7 — Prepare LungHist700 paths (20× only)

In [ ]:
# --- Cell 7: LungHist700 paths ---
LUNG_ROOT = Path('/content/drive/MyDrive/TFM/datasets/LungHist700')
assert LUNG_ROOT.exists(), f'LungHist700 not found at {LUNG_ROOT}'

LUNG_CLS_MAP = {'aca': 'lung adenocarcinoma', 'scc': 'lung squamous cell carcinoma', 'nor': 'benign lung tissue'}
lung_paths, lung_lc_indices = [], []
for p in sorted(LUNG_ROOT.rglob('*.jpg')):
    name = p.stem.lower()
    # e.g. 'aca_bd_20x_001'
    parts = re.split(r'[_-]', name)
    cls_tok = next((t for t in parts if t in LUNG_CLS_MAP), None)
    mag_tok = next((t for t in parts if t in ('20x', '40x')), None)
    if cls_tok and mag_tok == '20x':
        lc_name = LUNG_CLS_MAP[cls_tok]
        lc_id = next(c.id for c in CLASS_INFO if c.name == lc_name)
        lung_paths.append(str(p))
        lung_lc_indices.append(ID_TO_INDEX[lc_id])
print(f'LungHist700 (20×): {len(lung_paths)} patches')

## 8 — Encode all 4 datasets

In [ ]:
# --- Cell 8: encoding ---
def load_and_pre(path):
    img = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    return np.asarray(img, dtype=np.float32) / 255.0

def encode_paths(paths, batch=BATCH_SIZE, label=''):
    feats = []
    for start in range(0, len(paths), batch):
        batch_paths = paths[start:start+batch]
        arrs = np.stack([load_and_pre(p) for p in batch_paths])
        emb = model.encode_image(tf.constant(arrs), training=False).numpy()
        feats.append(emb)
        if start % (10 * batch) == 0:
            print(f'  {label}: {min(start+batch, len(paths))}/{len(paths)}')
    return np.concatenate(feats, axis=0)

print('Encoding LC25000 test...');    lc_feats   = encode_paths(lc_test_paths, label='LC25000')
print('Encoding NCT-CRC...');         nct_feats  = encode_paths(nct_paths,    label='NCT')
print('Encoding Chaoyang...');        chao_feats = encode_paths(chao_paths,   label='Chaoyang')
print('Encoding LungHist700...');     lung_feats = encode_paths(lung_paths,   label='LungHist700')

# Encode class prompts too (for anchor stars)
prompt_texts = [f'A histopathology image of {INDEX_TO_NAME[i]}.' for i in range(5)]
prompt_tokens = text_pre(prompt_texts)
prompt_feats = model.encode_text(prompt_tokens, training=False).numpy()
print(f'All shapes: LC={lc_feats.shape}, NCT={nct_feats.shape}, Chao={chao_feats.shape}, Lung={lung_feats.shape}, prompts={prompt_feats.shape}')

## 9 — Fit ONE shared UMAP

In [ ]:
# --- Cell 9: shared UMAP fit ---
import umap

all_feats = np.concatenate([lc_feats, nct_feats, chao_feats, lung_feats, prompt_feats], axis=0)
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=SEED)
all_2d = reducer.fit_transform(all_feats)

# Slice back
n_lc, n_nct, n_chao, n_lung = len(lc_feats), len(nct_feats), len(chao_feats), len(lung_feats)
lc_2d     = all_2d[:n_lc]
nct_2d    = all_2d[n_lc : n_lc + n_nct]
chao_2d   = all_2d[n_lc + n_nct : n_lc + n_nct + n_chao]
lung_2d   = all_2d[n_lc + n_nct + n_chao : n_lc + n_nct + n_chao + n_lung]
prompt_2d = all_2d[n_lc + n_nct + n_chao + n_lung :]
print('Shared UMAP fit done.')

## 10 — Render 2×2 scatter grid

In [ ]:
# --- Cell 10: 2x2 scatter grid ---
CLASS_COLOR = {
    0: '#2ca02c', 1: '#ff7f0e', 2: '#9467bd', 3: '#08519c', 4: '#a50f15',
}
CLASS_NAME_SHORT = {0: 'benign lung', 1: 'lung aca', 2: 'lung scc', 3: 'benign colon', 4: 'colon aca'}

lc_indices_arr   = np.array(lc_test_indices)
nct_indices_arr  = np.array(nct_lc_indices)
chao_indices_arr = np.array(chao_lc_indices)
lung_indices_arr = np.array(lung_lc_indices)

PANELS = [
    ('LC25000 (in-dist)',          None, None, None),
    ('LC25000 + NCT-CRC',          nct_2d, nct_indices_arr, 'NCT'),
    ('LC25000 + Chaoyang',         chao_2d, chao_indices_arr, 'Chaoyang'),
    ('LC25000 + LungHist700',      lung_2d, lung_indices_arr, 'LungHist700'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 12), dpi=120)
for ax, (title, ext_2d, ext_idx, ext_name) in zip(axes.flat, PANELS):
    # LC25000 dots (all 5 classes)
    for cls in range(5):
        m = lc_indices_arr == cls
        ax.scatter(lc_2d[m, 0], lc_2d[m, 1], s=6, c=CLASS_COLOR[cls], alpha=0.35, edgecolors='none')
    # External triangles
    if ext_2d is not None:
        for cls in range(5):
            m = ext_idx == cls
            if m.sum() == 0: continue
            ax.scatter(ext_2d[m, 0], ext_2d[m, 1], s=30, marker='^', c=CLASS_COLOR[cls],
                       edgecolors='black', linewidths=0.5, alpha=0.9)
    # Prompt anchor stars
    for cls in range(5):
        ax.scatter(prompt_2d[cls, 0], prompt_2d[cls, 1], s=200, marker='*',
                   c=CLASS_COLOR[cls], edgecolors='white', linewidths=1.5, zorder=6)
    ax.set_title(title, fontsize=14, fontweight='bold', color='#4D0912')
    ax.set_xticks([]); ax.set_yticks([])

# Legend below
legend_handles = [Patch(facecolor=CLASS_COLOR[i], label=CLASS_NAME_SHORT[i]) for i in range(5)]
fig.legend(handles=legend_handles, loc='lower center', ncol=5, frameon=False, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
out_scatter = '/content/baseline_shared_umap_2x2_scatter.png'
plt.savefig(out_scatter, dpi=120, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_scatter}')

## 11 — Render 2×2 thumbnails grid

Same panels but with tissue-image thumbnails overlaid on top of each cluster (LC25000 with thin borders, external with thick borders).

In [ ]:
# --- Cell 11: 2x2 thumbnails grid ---
def _sample_thumbs(coords, paths, indices_arr, n_per_class=3):
    """Return list of (x, y, thumbnail_array, class_id) for a subset of samples."""
    picks = []
    for cls in range(5):
        m = np.where(indices_arr == cls)[0]
        if len(m) == 0: continue
        chosen = np.random.default_rng(SEED).choice(m, size=min(n_per_class, len(m)), replace=False)
        for idx in chosen:
            try:
                thumb = Image.open(paths[idx]).convert('RGB').resize((48, 48))
                picks.append((coords[idx, 0], coords[idx, 1], np.array(thumb), cls))
            except Exception:
                pass
    return picks

fig, axes = plt.subplots(2, 2, figsize=(14, 12), dpi=120)
for ax, (title, ext_2d, ext_idx, ext_name) in zip(axes.flat, PANELS):
    # LC25000 thumbnails (thin borders)
    lc_picks = _sample_thumbs(lc_2d, lc_test_paths, lc_indices_arr, n_per_class=3)
    for x, y, arr, cls in lc_picks:
        oi = OffsetImage(arr, zoom=0.6)
        ab = AnnotationBbox(oi, (x, y), frameon=True,
                            bboxprops=dict(edgecolor=CLASS_COLOR[cls], linewidth=1.0))
        ax.add_artist(ab)
    # External thumbnails (thick borders)
    if ext_2d is not None:
        ext_paths = {'NCT': nct_paths, 'Chaoyang': chao_paths, 'LungHist700': lung_paths}[ext_name]
        ext_picks = _sample_thumbs(ext_2d, ext_paths, ext_idx, n_per_class=3)
        for x, y, arr, cls in ext_picks:
            oi = OffsetImage(arr, zoom=0.6)
            ab = AnnotationBbox(oi, (x, y), frameon=True,
                                bboxprops=dict(edgecolor=CLASS_COLOR[cls], linewidth=3.0))
            ax.add_artist(ab)
    # Set axis limits from the scatter version
    all_x = np.concatenate([lc_2d[:, 0]] + ([ext_2d[:, 0]] if ext_2d is not None else []))
    all_y = np.concatenate([lc_2d[:, 1]] + ([ext_2d[:, 1]] if ext_2d is not None else []))
    xpad = (all_x.max() - all_x.min()) * 0.05
    ypad = (all_y.max() - all_y.min()) * 0.05
    ax.set_xlim(all_x.min() - xpad, all_x.max() + xpad)
    ax.set_ylim(all_y.min() - ypad, all_y.max() + ypad)
    ax.set_title(title, fontsize=14, fontweight='bold', color='#4D0912')
    ax.set_xticks([]); ax.set_yticks([])

fig.legend(handles=legend_handles, loc='lower center', ncol=5, frameon=False, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
out_thumbs = '/content/baseline_shared_umap_2x2_thumbnails.png'
plt.savefig(out_thumbs, dpi=120, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_thumbs}')

## 12 — Download both PNGs

In [ ]:
# --- Cell 12: download ---
if IN_COLAB:
    files.download(out_scatter)
    files.download(out_thumbs)
print('Done. Send both files back to me and Ill slot them into slide 14.')